# Contrastive Learning for Embeddings

## Historical Context

Contrastive learning revolutionized representation learning:

- **2018**: Triplet loss for face recognition
- **2020**: SimCLR (Google) - contrastive learning for images
- **2020**: MoCo (Facebook) - momentum contrast
- **2021**: SimCSE - contrastive learning for sentence embeddings
- **2021**: Sentence Transformers become popular
- **2022**: E5 embeddings (Microsoft) - large-scale contrastive training
- **2023**: Instructor embeddings - task-specific instructions
- **2023**: BGE embeddings (BAAI) - state-of-the-art retrieval
- **2024**: Matryoshka embeddings - flexible dimensions
- **2025**: Contrastive learning standard for embedding models

## Why Contrastive Learning?

**Goal**: Learn embeddings where similar items are close, dissimilar items are far

**Key idea**: 
- Pull positive pairs together
- Push negative pairs apart

**Applications**:
- Semantic search
- Document retrieval
- Question answering
- Duplicate detection
- Recommendation systems
- Zero-shot classification

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass

## 1. Contrastive Loss Functions

In [ ]:
class TripletLoss(nn.Module):
    """Triplet loss: anchor, positive, negative.
    
    Loss = max(0, d(a, p) - d(a, n) + margin)
    
    Where:
    - a: anchor embedding
    - p: positive (similar to anchor)
    - n: negative (dissimilar to anchor)
    - margin: minimum separation between positive and negative
    
    Used in: Face recognition, person re-identification
    """
    
    def __init__(self, margin: float = 1.0, distance: str = 'euclidean'):
        super().__init__()
        self.margin = margin
        self.distance = distance
    
    def forward(self, anchor: torch.Tensor, positive: torch.Tensor, 
                negative: torch.Tensor) -> torch.Tensor:
        """Compute triplet loss.
        
        Args:
            anchor: [batch_size, embedding_dim]
            positive: [batch_size, embedding_dim]
            negative: [batch_size, embedding_dim]
        """
        if self.distance == 'euclidean':
            pos_dist = F.pairwise_distance(anchor, positive, p=2)
            neg_dist = F.pairwise_distance(anchor, negative, p=2)
        elif self.distance == 'cosine':
            pos_dist = 1 - F.cosine_similarity(anchor, positive)
            neg_dist = 1 - F.cosine_similarity(anchor, negative)
        else:
            raise ValueError(f"Unknown distance: {self.distance}")
        
        # Triplet loss: encourage pos_dist < neg_dist - margin
        loss = F.relu(pos_dist - neg_dist + self.margin)
        return loss.mean()


class InfoNCELoss(nn.Module):
    """InfoNCE (Normalized Temperature-scaled Cross Entropy).
    
    Also known as NT-Xent loss, used in SimCLR.
    
    Key idea:
    - For each anchor, predict which of N candidates is the positive
    - All other samples in batch are negatives (in-batch negatives)
    
    Advantages:
    - No explicit negative sampling needed
    - Uses all batch samples efficiently
    - Scales well with batch size
    
    Used by: SimCLR, CLIP, sentence transformers
    """
    
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, anchor: torch.Tensor, positive: torch.Tensor) -> torch.Tensor:
        """Compute InfoNCE loss with in-batch negatives.
        
        Args:
            anchor: [batch_size, embedding_dim]
            positive: [batch_size, embedding_dim]
        """
        batch_size = anchor.size(0)
        
        # Normalize embeddings
        anchor = F.normalize(anchor, dim=1)
        positive = F.normalize(positive, dim=1)
        
        # Compute similarity matrix: [batch_size, batch_size]
        # anchor[i] vs all positives
        similarity = torch.matmul(anchor, positive.T) / self.temperature
        
        # Labels: positive for each anchor is at index i
        labels = torch.arange(batch_size, device=anchor.device)
        
        # Cross-entropy loss
        loss = F.cross_entropy(similarity, labels)
        
        return loss


class MultipleNegativesRankingLoss(nn.Module):
    """Multiple Negatives Ranking Loss (used in sentence transformers).
    
    Similar to InfoNCE but with explicit query/document separation.
    Common in information retrieval.
    
    For each query:
    - One positive document
    - All other documents in batch are negatives
    """
    
    def __init__(self, scale: float = 20.0):
        super().__init__()
        self.scale = scale
        self.cross_entropy = nn.CrossEntropyLoss()
    
    def forward(self, query_embeddings: torch.Tensor, 
                doc_embeddings: torch.Tensor) -> torch.Tensor:
        """Compute loss.
        
        Args:
            query_embeddings: [batch_size, embedding_dim]
            doc_embeddings: [batch_size, embedding_dim]
        """
        # Normalize
        query_embeddings = F.normalize(query_embeddings, dim=1)
        doc_embeddings = F.normalize(doc_embeddings, dim=1)
        
        # Similarity scores: [batch_size, batch_size]
        scores = torch.matmul(query_embeddings, doc_embeddings.T) * self.scale
        
        # Labels: diagonal is positive
        labels = torch.arange(query_embeddings.size(0), device=query_embeddings.device)
        
        return self.cross_entropy(scores, labels)


class CosineSimilarityLoss(nn.Module):
    """Simple cosine similarity loss (used in SimCSE).
    
    Minimizes cosine distance between anchor and positive.
    Relies on in-batch negatives for separation.
    """
    
    def __init__(self):
        super().__init__()
    
    def forward(self, anchor: torch.Tensor, positive: torch.Tensor,
                negative: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Compute loss."""
        # Positive pair loss
        pos_loss = 1 - F.cosine_similarity(anchor, positive).mean()
        
        if negative is not None:
            # Negative pair loss (push apart)
            neg_loss = F.cosine_similarity(anchor, negative).mean()
            return pos_loss + neg_loss
        
        return pos_loss


# Demonstrate loss functions
def compare_loss_functions(batch_size: int = 32, embedding_dim: int = 128):
    """Compare different contrastive loss functions."""
    
    # Generate synthetic embeddings
    torch.manual_seed(42)
    anchor = torch.randn(batch_size, embedding_dim)
    positive = anchor + 0.1 * torch.randn(batch_size, embedding_dim)  # Similar
    negative = torch.randn(batch_size, embedding_dim)  # Dissimilar
    
    # Compute losses
    losses = {
        'Triplet': TripletLoss(margin=1.0)(anchor, positive, negative),
        'InfoNCE': InfoNCELoss(temperature=0.07)(anchor, positive),
        'MNR': MultipleNegativesRankingLoss(scale=20.0)(anchor, positive),
        'Cosine': CosineSimilarityLoss()(anchor, positive, negative),
    }
    
    print("Contrastive Loss Comparison")
    print("="*60)
    for name, loss in losses.items():
        print(f"{name:20s}: {loss.item():.4f}")
    
    # Visualize
    plt.figure(figsize=(10, 6))
    names = list(losses.keys())
    values = [losses[name].item() for name in names]
    bars = plt.bar(names, values, edgecolor='black', alpha=0.7)
    plt.ylabel('Loss Value', fontsize=12)
    plt.title('Contrastive Loss Functions', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}',
                ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig('contrastive_losses.png', dpi=150, bbox_inches='tight')
    plt.show()

compare_loss_functions()

## 2. SimCSE: Simple Contrastive Sentence Embeddings

In [ ]:
class SimCSE(nn.Module):
    """SimCSE: Simple Contrastive Learning of Sentence Embeddings (Gao et al., 2021).
    
    Key insight: Use dropout as data augmentation!
    - Pass same sentence through model twice with different dropout masks
    - Two representations are positive pair
    - Other sentences in batch are negatives
    
    Unsupervised SimCSE:
    - No labeled data needed
    - Just sentences
    
    Supervised SimCSE:
    - Use NLI datasets (entailment = positive, contradiction = hard negative)
    """
    
    def __init__(self, encoder: nn.Module, pooling: str = 'cls'):
        super().__init__()
        self.encoder = encoder
        self.pooling = pooling
        
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """Encode sentences."""
        # Get encoder output
        # In real implementation, would use actual transformer
        # Here we simulate with random embeddings
        batch_size, seq_len = input_ids.shape
        hidden_size = 768
        hidden_states = torch.randn(batch_size, seq_len, hidden_size)
        
        # Pooling
        if self.pooling == 'cls':
            # Use [CLS] token
            embeddings = hidden_states[:, 0, :]
        elif self.pooling == 'mean':
            # Mean pooling (with attention mask)
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.shape).float()
            sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            embeddings = sum_embeddings / sum_mask
        else:
            raise ValueError(f"Unknown pooling: {self.pooling}")
        
        return embeddings
    
    def unsupervised_loss(self, input_ids: torch.Tensor, 
                         attention_mask: torch.Tensor,
                         temperature: float = 0.05) -> torch.Tensor:
        """Compute unsupervised SimCSE loss.
        
        Pass each sentence through encoder twice with different dropout.
        """
        # First forward pass
        embeddings1 = self.forward(input_ids, attention_mask)
        
        # Second forward pass (different dropout)
        embeddings2 = self.forward(input_ids, attention_mask)
        
        # InfoNCE loss
        loss_fn = InfoNCELoss(temperature=temperature)
        loss = loss_fn(embeddings1, embeddings2)
        
        return loss
    
    def supervised_loss(self, anchor_ids: torch.Tensor, anchor_mask: torch.Tensor,
                       positive_ids: torch.Tensor, positive_mask: torch.Tensor,
                       negative_ids: torch.Tensor, negative_mask: torch.Tensor,
                       temperature: float = 0.05) -> torch.Tensor:
        """Compute supervised SimCSE loss with hard negatives."""
        # Encode all
        anchor_emb = self.forward(anchor_ids, anchor_mask)
        positive_emb = self.forward(positive_ids, positive_mask)
        negative_emb = self.forward(negative_ids, negative_mask)
        
        # Normalize
        anchor_emb = F.normalize(anchor_emb, dim=1)
        positive_emb = F.normalize(positive_emb, dim=1)
        negative_emb = F.normalize(negative_emb, dim=1)
        
        # Compute similarities
        pos_sim = torch.sum(anchor_emb * positive_emb, dim=1) / temperature
        neg_sim = torch.sum(anchor_emb * negative_emb, dim=1) / temperature
        
        # Contrastive loss
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1)
        labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
        loss = F.cross_entropy(logits, labels)
        
        return loss


print("SimCSE: Contrastive Learning with Dropout")
print("="*80)
print("\nKey Innovation:")
print("  - Use dropout as data augmentation (no external augmentation needed)")
print("  - Same sentence, different dropout masks = positive pair")
print("  - Other sentences in batch = negatives")
print("\nResults (from paper):")
print("  - Unsupervised SimCSE: 76.3 → 81.6 on STS (semantic textual similarity)")
print("  - Supervised SimCSE: 81.6 → 84.9 (with NLI data)")
print("  - Simple yet highly effective!")

## 3. Sentence Transformers: Practical Embeddings

In [ ]:
class SentenceTransformer(nn.Module):
    """Sentence Transformer architecture (Reimers & Gurevych, 2019).
    
    Standard architecture for sentence embeddings:
    1. Transformer encoder (BERT, RoBERTa, etc.)
    2. Pooling layer (mean, max, CLS)
    3. Optional projection layer
    
    Training strategies:
    - Contrastive learning
    - Triplet loss
    - Multiple negatives ranking
    - In-batch negatives
    """
    
    def __init__(self, encoder: nn.Module, embedding_dim: int = 768,
                 pooling_mode: str = 'mean', normalize: bool = True):
        super().__init__()
        self.encoder = encoder
        self.pooling_mode = pooling_mode
        self.normalize = normalize
        self.embedding_dim = embedding_dim
    
    def mean_pooling(self, token_embeddings: torch.Tensor, 
                    attention_mask: torch.Tensor) -> torch.Tensor:
        """Mean pooling with attention mask."""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.shape).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = input_mask_expanded.sum(dim=1).clamp(min=1e-9)
        return sum_embeddings / sum_mask
    
    def encode(self, input_ids: torch.Tensor, 
              attention_mask: torch.Tensor) -> torch.Tensor:
        """Encode text to embeddings."""
        # Simulate encoder output
        batch_size, seq_len = input_ids.shape
        token_embeddings = torch.randn(batch_size, seq_len, self.embedding_dim)
        
        # Pooling
        if self.pooling_mode == 'mean':
            embeddings = self.mean_pooling(token_embeddings, attention_mask)
        elif self.pooling_mode == 'cls':
            embeddings = token_embeddings[:, 0, :]
        elif self.pooling_mode == 'max':
            embeddings = torch.max(token_embeddings, dim=1)[0]
        else:
            raise ValueError(f"Unknown pooling mode: {self.pooling_mode}")
        
        # Normalize
        if self.normalize:
            embeddings = F.normalize(embeddings, p=2, dim=1)
        
        return embeddings


@dataclass
class TrainingExample:
    """Training example for sentence embeddings."""
    query: str
    positive: str
    negatives: List[str]


def create_embedding_dataset() -> List[TrainingExample]:
    """Create sample dataset for embedding training."""
    examples = [
        TrainingExample(
            query="What is machine learning?",
            positive="Machine learning is a subset of AI that learns from data.",
            negatives=[
                "The weather is nice today.",
                "I like pizza for dinner.",
            ]
        ),
        TrainingExample(
            query="How to train a neural network?",
            positive="Neural networks are trained using backpropagation and gradient descent.",
            negatives=[
                "The Eiffel Tower is in Paris.",
                "Water boils at 100 degrees Celsius.",
            ]
        ),
    ]
    return examples


# Demonstrate sentence transformer usage
print("Sentence Transformer Architecture")
print("="*80)

examples = create_embedding_dataset()
print(f"\nCreated {len(examples)} training examples")
print("\nExample:")
ex = examples[0]
print(f"  Query: {ex.query}")
print(f"  Positive: {ex.positive}")
print(f"  Negatives: {ex.negatives}")

print("\nPopular Sentence Transformer Models (HuggingFace):")
models = {
    'all-MiniLM-L6-v2': 'Fast and good quality (384 dim)',
    'all-mpnet-base-v2': 'Best quality (768 dim)',
    'multi-qa-mpnet-base-dot-v1': 'Optimized for QA',
    'paraphrase-multilingual-mpnet-base-v2': 'Multilingual (50+ languages)',
    'BAAI/bge-base-en-v1.5': 'SOTA retrieval (2023)',
    'intfloat/e5-base-v2': 'E5 embeddings (Microsoft)',
}

for name, desc in models.items():
    print(f"  {name:45s}: {desc}")

## 4. Hard Negative Mining

In [ ]:
class HardNegativeMiner:
    """Mine hard negatives for better contrastive learning.
    
    Hard negatives: negatives that are similar to query but not relevant.
    These are more informative than random negatives.
    
    Strategies:
    1. Random negatives (easy, but less effective)
    2. Hard negatives from same batch (in-batch hard negatives)
    3. Hard negatives from retrieval (BM25, dense retrieval)
    4. Cross-batch hard negatives (from previous batches)
    """
    
    @staticmethod
    def mine_in_batch_hard_negatives(
        query_embeddings: torch.Tensor,
        doc_embeddings: torch.Tensor,
        num_hard_negatives: int = 3
    ) -> torch.Tensor:
        """Find hard negatives within current batch.
        
        For each query, find most similar documents (excluding ground truth).
        """
        batch_size = query_embeddings.size(0)
        
        # Compute similarity matrix
        similarities = torch.matmul(query_embeddings, doc_embeddings.T)
        
        # Mask out ground truth (diagonal)
        mask = torch.eye(batch_size, device=similarities.device).bool()
        similarities = similarities.masked_fill(mask, float('-inf'))
        
        # Get top-k most similar (hard negatives)
        _, hard_negative_indices = torch.topk(
            similarities, k=num_hard_negatives, dim=1
        )
        
        return hard_negative_indices
    
    @staticmethod
    def compute_loss_with_hard_negatives(
        query_embeddings: torch.Tensor,
        doc_embeddings: torch.Tensor,
        temperature: float = 0.05,
        num_hard_negatives: int = 3
    ) -> torch.Tensor:
        """Compute contrastive loss with hard negative mining."""
        batch_size = query_embeddings.size(0)
        
        # Normalize
        query_embeddings = F.normalize(query_embeddings, dim=1)
        doc_embeddings = F.normalize(doc_embeddings, dim=1)
        
        # Compute full similarity matrix
        similarities = torch.matmul(query_embeddings, doc_embeddings.T) / temperature
        
        # Ground truth: diagonal
        labels = torch.arange(batch_size, device=similarities.device)
        
        # Basic loss (all in-batch negatives)
        loss = F.cross_entropy(similarities, labels)
        
        return loss


class CrossBatchNegatives:
    """Maintain queue of embeddings from previous batches.
    
    Increases effective batch size for negative sampling without
    increasing memory requirements.
    
    Used in MoCo (Momentum Contrast).
    """
    
    def __init__(self, queue_size: int = 65536, embedding_dim: int = 768):
        self.queue_size = queue_size
        self.embedding_dim = embedding_dim
        
        # Initialize queue
        self.register_buffer(
            'queue',
            torch.randn(embedding_dim, queue_size)
        )
        self.queue = F.normalize(self.queue, dim=0)
        self.register_buffer('queue_ptr', torch.zeros(1, dtype=torch.long))
    
    def register_buffer(self, name: str, tensor: torch.Tensor):
        """Register buffer (simplified)."""
        setattr(self, name, tensor)
    
    @torch.no_grad()
    def update_queue(self, embeddings: torch.Tensor):
        """Add new embeddings to queue, remove oldest."""
        batch_size = embeddings.size(0)
        
        ptr = int(self.queue_ptr)
        
        # Replace oldest embeddings
        if ptr + batch_size <= self.queue_size:
            self.queue[:, ptr:ptr + batch_size] = embeddings.T
        else:
            # Wrap around
            remaining = self.queue_size - ptr
            self.queue[:, ptr:] = embeddings[:remaining].T
            self.queue[:, :batch_size - remaining] = embeddings[remaining:].T
        
        # Update pointer
        self.queue_ptr[0] = (ptr + batch_size) % self.queue_size
    
    def get_negatives(self) -> torch.Tensor:
        """Get all embeddings in queue as negatives."""
        return self.queue.clone().T


# Demonstrate hard negative mining
def demo_hard_negative_mining(batch_size: int = 32, embedding_dim: int = 128):
    """Demonstrate hard negative mining impact."""
    
    torch.manual_seed(42)
    
    # Generate synthetic embeddings
    query_embeddings = torch.randn(batch_size, embedding_dim)
    doc_embeddings = torch.randn(batch_size, embedding_dim)
    
    # Normalize
    query_embeddings = F.normalize(query_embeddings, dim=1)
    doc_embeddings = F.normalize(doc_embeddings, dim=1)
    
    # Mine hard negatives
    miner = HardNegativeMiner()
    hard_neg_indices = miner.mine_in_batch_hard_negatives(
        query_embeddings, doc_embeddings, num_hard_negatives=5
    )
    
    # Compute similarity matrix
    similarities = torch.matmul(query_embeddings, doc_embeddings.T)
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Full similarity matrix
    im1 = ax1.imshow(similarities.numpy(), cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
    ax1.set_xlabel('Document Index', fontsize=12)
    ax1.set_ylabel('Query Index', fontsize=12)
    ax1.set_title('Query-Document Similarity Matrix', fontsize=14, fontweight='bold')
    plt.colorbar(im1, ax=ax1)
    
    # Highlight hard negatives for first query
    similarities_copy = similarities.clone()
    hard_mask = torch.zeros_like(similarities_copy)
    hard_mask[0, hard_neg_indices[0]] = 1
    
    im2 = ax2.imshow(similarities_copy[0].numpy(), cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
    ax2.scatter(hard_neg_indices[0].numpy(), [0]*len(hard_neg_indices[0]), 
               c='red', s=200, marker='x', linewidths=3, label='Hard Negatives')
    ax2.scatter([0], [0], c='green', s=200, marker='o', linewidths=3, label='Positive')
    ax2.set_xlabel('Document Index', fontsize=12)
    ax2.set_ylabel('Similarity', fontsize=12)
    ax2.set_title('Hard Negatives for Query 0', fontsize=14, fontweight='bold')
    ax2.legend()
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.savefig('hard_negative_mining.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Hard Negative Mining")
    print("="*80)
    print(f"\nQuery 0 similarities:")
    print(f"  Positive (doc 0): {similarities[0, 0].item():.4f}")
    print(f"  Hard negatives: {similarities[0, hard_neg_indices[0]].numpy()}")
    print(f"  Mean random negative: {similarities[0].mean().item():.4f}")
    print(f"\nHard negatives are more challenging (higher similarity to query)")

demo_hard_negative_mining()

## 5. Training Pipeline for Embeddings

In [ ]:
class EmbeddingTrainer:
    """Complete training pipeline for embedding models."""
    
    def __init__(self, model: nn.Module, config: Dict):
        self.model = model
        self.config = config
        
        # Loss function
        self.loss_fn = MultipleNegativesRankingLoss(scale=20.0)
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.get('learning_rate', 2e-5),
            weight_decay=config.get('weight_decay', 0.01)
        )
    
    def train_step(self, query_batch: Dict, doc_batch: Dict) -> float:
        """Single training step."""
        self.model.train()
        
        # Encode queries and documents
        query_embeddings = self.model.encode(
            query_batch['input_ids'],
            query_batch['attention_mask']
        )
        doc_embeddings = self.model.encode(
            doc_batch['input_ids'],
            doc_batch['attention_mask']
        )
        
        # Compute loss
        loss = self.loss_fn(query_embeddings, doc_embeddings)
        
        # Backward pass
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
    
    @torch.no_grad()
    def evaluate(self, eval_dataloader) -> Dict[str, float]:
        """Evaluate on retrieval task."""
        self.model.eval()
        
        # Compute metrics
        # In practice: recall@k, MRR, NDCG
        metrics = {
            'recall@1': 0.75,
            'recall@10': 0.92,
            'mrr': 0.83,
            'ndcg@10': 0.88,
        }
        
        return metrics


def simulate_training(num_steps: int = 1000):
    """Simulate embedding model training."""
    
    print("Embedding Model Training Simulation")
    print("="*80)
    
    # Setup (simplified)
    model = SentenceTransformer(
        encoder=None,  # Would use actual BERT
        embedding_dim=768,
        pooling_mode='mean'
    )
    
    config = {
        'learning_rate': 2e-5,
        'batch_size': 32,
        'weight_decay': 0.01,
    }
    
    trainer = EmbeddingTrainer(model, config)
    
    # Simulate training curve
    losses = []
    for step in range(num_steps):
        # Simulate decreasing loss
        loss = 5.0 * np.exp(-step / 200) + 0.5 + 0.1 * np.random.randn()
        losses.append(max(loss, 0.3))  # Floor
        
        if step % 100 == 0:
            print(f"Step {step:4d} | Loss: {losses[-1]:.4f}")
    
    # Plot training curve
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Loss curve
    ax1.plot(losses, alpha=0.6, label='Raw loss')
    # Smoothed
    window = 50
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, num_steps), smoothed, linewidth=2, label='Smoothed', color='red')
    ax1.set_xlabel('Training Step', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Metrics progression
    steps_eval = [0, 200, 400, 600, 800, 1000]
    recall_1 = [0.45, 0.58, 0.67, 0.72, 0.75, 0.77]
    recall_10 = [0.72, 0.81, 0.87, 0.90, 0.92, 0.93]
    
    ax2.plot(steps_eval, recall_1, marker='o', linewidth=2, markersize=8, label='Recall@1')
    ax2.plot(steps_eval, recall_10, marker='s', linewidth=2, markersize=8, label='Recall@10')
    ax2.set_xlabel('Training Step', fontsize=12)
    ax2.set_ylabel('Recall', fontsize=12)
    ax2.set_title('Retrieval Performance', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig('embedding_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nFinal metrics (step {num_steps}):")
    print(f"  Recall@1: {recall_1[-1]:.2%}")
    print(f"  Recall@10: {recall_10[-1]:.2%}")

simulate_training()

## Key Takeaways

### Contrastive Learning Basics
- **Goal**: Learn embeddings where similar items close, dissimilar items far
- **Key insight**: Pull positive pairs together, push negatives apart
- **Applications**: Search, retrieval, QA, recommendations

### Loss Functions

**Triplet Loss**:
- (anchor, positive, negative) triplets
- Requires explicit negative sampling
- Classic but effective

**InfoNCE Loss**:
- Uses in-batch negatives (efficient!)
- Temperature-scaled
- Used by SimCLR, CLIP

**Multiple Negatives Ranking**:
- Standard for sentence transformers
- Query-document pairs
- All other docs are negatives

### SimCSE Innovation
- **Unsupervised**: Use dropout as augmentation
- **Supervised**: Use NLI data (entailment/contradiction)
- **Results**: 76.3 → 84.9 on STS benchmarks
- **Simple yet powerful**

### Hard Negative Mining

**Why hard negatives?**
- Random negatives too easy
- Hard negatives more informative
- Faster convergence, better quality

**Strategies**:
1. In-batch hard negatives
2. BM25 retrieval
3. Cross-batch negatives (queue)
4. Hard negative mining from previous model

### Training Best Practices

**Data**:
- Diverse query-document pairs
- Mix of easy and hard negatives
- 100K-1M pairs for good quality

**Hyperparameters**:
- LR: 2e-5 (same as BERT fine-tuning)
- Batch size: 32-128 (larger better for negatives)
- Temperature: 0.05-0.1
- Training: 1-3 epochs

**Architecture**:
- Mean pooling usually best
- Normalize embeddings
- 768 dim (BERT) or 384 dim (distilled)

### Popular Models (2024-2025)

1. **BGE (BAAI)**: State-of-the-art retrieval
2. **E5 (Microsoft)**: Instruction-based embeddings
3. **Instructor**: Task-specific instructions
4. **GTE**: General text embeddings
5. **Sentence-T5**: T5-based embeddings

### Evaluation Metrics

**Retrieval**:
- Recall@k (k=1, 5, 10, 100)
- MRR (Mean Reciprocal Rank)
- NDCG@k (Normalized Discounted Cumulative Gain)

**Similarity**:
- STS (Semantic Textual Similarity)
- Spearman correlation

### Production Tips

1. **Start with pretrained** (e.g., sentence-transformers)
2. **Fine-tune on domain data** (1K-10K pairs)
3. **Use hard negatives** (mine from BM25)
4. **Larger batch = better** (more negatives)
5. **Evaluate on retrieval**, not just similarity
6. **Consider dimensionality** (384 vs 768 vs 1024)
7. **Normalize embeddings** for cosine similarity
8. **Index with FAISS/Milvus** for production